In [1]:
# data_generator_tensor.py
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ================= 配置区域 =================
# [修改] 数据集后缀改为 Tensor
DATASET_ROOT = "./dataset_tri_RAD_Tensor2"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

# 真实数据路径
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/L20250425_103838_F6000000000_S400000_B100000_M0_IQ_ID0_nozaoshen_100ksps_QPSK.dat": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/L20250425_104306_F6000000000_S400000_B100000_M0_IQ_ID0_nojiazao_100ksps_OQPSK.dat": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}
#"E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK"
#"E:/Project/AMR-Benchmark/OQPSK-03-24-25-10h54m02s6.iq": "OQPSK"

SAMPLES_PER_IMAGE = 1024
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 800
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 核心处理逻辑 =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_tensor_data(iq_complex_data, img_size=64):
    """
    生成纯数据张量，不转换成图片格式。
    Output: Tensor shape (3, 64, 64), dtype=float32, range approx [0, 1]
    """
    if iq_complex_data.dim() == 1:
        iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    
    if iq_data.numel() < 100: 
        return torch.zeros(3, img_size, img_size)

    # 归一化
    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    # 提取特征
    real = norm_iq.real
    imag = norm_iq.imag
    phase = torch.angle(norm_iq) / np.pi 
    mag = torch.abs(norm_iq)
    
    diff_signal = norm_iq[1:] * torch.conj(norm_iq[:-1])
    d_scale = torch.sqrt(torch.mean(diff_signal.real**2 + diff_signal.imag**2)) + 1e-9
    diff_signal = diff_signal / (3.0 * d_scale)
    d_real = diff_signal.real
    d_imag = diff_signal.imag
    
    # 生成直方图
    bins = img_size
    hist_r = fast_hist2d(real, imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    hist_a = fast_hist2d(phase, mag, bins=bins, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    hist_d = fast_hist2d(d_real, d_imag, bins=bins, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    # 数值处理 (Log压缩 + 归一化)
    # 保持为 float32，不转 int
    def process(h):
        h = torch.log1p(h)
        mx = h.max()
        if mx > 1e-6: h = h / mx
        return h

    # 堆叠成 (3, 64, 64)
    tensor = torch.stack([process(hist_r), process(hist_a), process(hist_d)], dim=0)
    
    # 翻转Y轴符合视觉习惯
    return torch.flip(tensor, [1])

# ================= 信号仿真 =================
def generate_sim_signal(mod_type, N, snr_db):

    pass

# ================= 执行生成 =================
if __name__ == "__main__":
    # data_generator_rad.py文件导入generate_sim_signal
    try:
        from data_generator_rad import generate_sim_signal
    except ImportError:
        print("请确保 generate_sim_signal 函数已定义！")

    if os.path.exists(DATASET_ROOT):
        shutil.rmtree(DATASET_ROOT)
    
    print("===== Generating Pure Tensor Data (.pt) =====")
    print("This will save (3, 64, 64) float tensors directly.")
    
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        
        print(f"Generating {cls_name}...")
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            
            snr = np.random.uniform(-5, 25)
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            
            # 生成 Tensor
            tensor_data = generate_tensor_data(sig.to(DEVICE), IMAGE_SIZE)
            
            # [关键] 保存为 .pt 文件
            save_path = os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.pt")
            torch.save(tensor_data.cpu(), save_path) # 存到CPU以节省空间
            
            # (可选) 仅保存第一张为图片，方便人类查看数据长什么样
            if i == 0:
                from PIL import Image
                arr = (tensor_data.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                Image.fromarray(arr).save(os.path.join(target_dir, "viz_sample.png"))

    print("\nProcessing Real Data...")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): continue
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            raw = np.fromfile(file_path, dtype=np.int16)
            if len(raw) % 2 != 0: raw = raw[:-1]
            iq_complex = (raw[::2] + 1j * raw[1::2]).astype(np.complex64) / 32767.0
            
            num_slices = len(iq_complex) // SAMPLES_PER_IMAGE
            for i in range(num_slices):
                chunk = torch.from_numpy(iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]).to(DEVICE)
                tensor_data = generate_tensor_data(chunk, IMAGE_SIZE)
                
                # 保存 .pt
                torch.save(tensor_data.cpu(), os.path.join(save_dir, f"real_{i:04d}.pt"))
        except Exception as e:
            print(f"Error {label}: {e}")
            
    print(f"\nDone! Dataset saved at {DATASET_ROOT}")

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===== Generating Pure Tensor Data (.pt) =====
This will save (3, 64, 64) float tensors directly.
Generating BPSK...


100%|██████████| 800/800 [00:03<00:00, 255.05it/s]


Generating QPSK...


100%|██████████| 800/800 [00:01<00:00, 536.74it/s]


Generating 8PSK...


... (16 outputs left out)

100%|██████████| 800/800 [00:01<00:00, 413.55it/s]


Generating FSK2...


100%|██████████| 800/800 [00:01<00:00, 489.14it/s]



Processing Real Data...

Done! Dataset saved at ./dataset_tri_RAD_Tensor2
